In [4]:
import tomllib
from pathlib import Path
import psycopg2

with open("../../../.streamlit/secrets.toml", "rb") as file:
    secrets = tomllib.load(file)

database = secrets["database"]

connection = psycopg2.connect(
    dbname=database["database"],
    user=database["username"],
    password=database["password"],
    host=database["host"],
    port=database["port"],
    sslmode=database["sslmode"],
)

print("Connected to PostgreSQL")

Connected to PostgreSQL


In [5]:
create_table_query = """
CREATE TABLE ai_usage_records (
    id BIGSERIAL PRIMARY KEY,
    operation TEXT NOT NULL,
    model TEXT NOT NULL,
    prompt_tokens INTEGER NOT NULL,
    completion_tokens INTEGER NOT NULL,
    total_tokens INTEGER NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    user_id TEXT,
    project_id TEXT
);
"""

with connection.cursor() as cursor:
    cursor.execute(create_table_query)

connection.commit()

print("Table created successfully")

Table created successfully


In [6]:
with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM ai_usage_records LIMIT 0")
    columns = [column.name for column in cursor.description]

print(columns)

connection.close()

['id', 'operation', 'model', 'prompt_tokens', 'completion_tokens', 'total_tokens', 'created_at', 'user_id', 'project_id']


In [3]:
migration_query = """
ALTER TABLE ai_usage_records
    ADD COLUMN input_tokens INTEGER,
    ADD COLUMN output_tokens INTEGER,
    ADD COLUMN input_text_tokens INTEGER,
    ADD COLUMN input_image_tokens INTEGER,
    ADD COLUMN output_text_tokens INTEGER,
    ADD COLUMN output_image_tokens INTEGER;

UPDATE ai_usage_records
SET
    input_tokens = prompt_tokens,
    output_tokens = completion_tokens;

ALTER TABLE ai_usage_records
    ALTER COLUMN input_tokens SET NOT NULL,
    ALTER COLUMN output_tokens SET NOT NULL;

ALTER TABLE ai_usage_records
    DROP COLUMN prompt_tokens,
    DROP COLUMN completion_tokens;
"""

with connection:
    with connection.cursor() as cursor:
        cursor.execute(migration_query)

print("AI usage table migrated successfully.")

AI usage table migrated successfully.


In [4]:
with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM ai_usage_records LIMIT 0")
    columns = [column.name for column in cursor.description]

print(columns)

['id', 'operation', 'model', 'total_tokens', 'created_at', 'user_id', 'project_id', 'input_tokens', 'output_tokens', 'input_text_tokens', 'input_image_tokens', 'output_text_tokens', 'output_image_tokens']


In [10]:
with connection.cursor() as cursor:
    cursor.execute("""
        SELECT
            id,
            operation,
            model,
            input_tokens,
            output_tokens,
            total_tokens,
            project_id,
            created_at
        FROM ai_usage_records
        WHERE project_id = '2'
        ORDER BY created_at DESC;
    """)

    records = cursor.fetchall()

    for record in records:
        print(record)

connection.close()

(30, 'generate_scene_illustration', 'gpt-image-2.5-flare', 836, 439, 1275, '2', datetime.datetime(2026, 11, 19, 17, 2, 20, 34321, tzinfo=datetime.timezone.utc))
(29, 'generate_scene_illustration', 'gpt-image-2.5-flare', 783, 439, 1222, '2', datetime.datetime(2026, 11, 19, 17, 2, 4, 635513, tzinfo=datetime.timezone.utc))
(28, 'generate_scene_illustration', 'gpt-image-2.5-flare', 718, 439, 1157, '2', datetime.datetime(2026, 11, 19, 17, 1, 47, 915940, tzinfo=datetime.timezone.utc))
(27, 'generate_initial_story', 'gpt-4o-2024-08-06', 1410, 1632, 3042, '2', datetime.datetime(2026, 11, 19, 17, 1, 33, 365164, tzinfo=datetime.timezone.utc))
(26, 'generate_story_blueprint', 'gpt-4o-2024-08-06', 239, 664, 903, '2', datetime.datetime(2026, 11, 19, 17, 1, 12, 713941, tzinfo=datetime.timezone.utc))
(25, 'generate_initial_story', 'gpt-4o-2024-08-06', 4287, 2166, 6453, '2', datetime.datetime(2026, 11, 19, 16, 57, 38, 904971, tzinfo=datetime.timezone.utc))
(24, 'generate_story_blueprint', 'gpt-4o-2024

In [9]:
connection.rollback()

In [ ]:

schema_query = """
-- ============================================================
-- STORIES
-- ============================================================

CREATE TABLE stories (
    id TEXT PRIMARY KEY,
    title TEXT NOT NULL,
    genre TEXT,
    tone TEXT,
    length TEXT,
    creativity FLOAT
);

-- ============================================================
-- STORY ELEMENTS
-- ============================================================

CREATE TABLE worlds (
    id TEXT PRIMARY KEY,
    story_id TEXT NOT NULL UNIQUE REFERENCES stories(id) ON DELETE CASCADE,
    name TEXT NOT NULL,
    description TEXT,
    source_library_item_id TEXT
);

CREATE TABLE characters (
    id TEXT PRIMARY KEY,
    story_id TEXT NOT NULL REFERENCES stories(id) ON DELETE CASCADE,
    name TEXT NOT NULL,
    description TEXT,
    role TEXT,
    source_library_item_id TEXT
);

CREATE TABLE locations (
    id TEXT PRIMARY KEY,
    story_id TEXT NOT NULL REFERENCES stories(id) ON DELETE CASCADE,
    name TEXT NOT NULL,
    description TEXT,
    source_library_item_id TEXT
);

CREATE TABLE artifacts (
    id TEXT PRIMARY KEY,
    story_id TEXT NOT NULL REFERENCES stories(id) ON DELETE CASCADE,
    name TEXT NOT NULL,
    description TEXT,
    source_library_item_id TEXT
);

CREATE TABLE relationships (
    id TEXT PRIMARY KEY,
    story_id TEXT NOT NULL REFERENCES stories(id) ON DELETE CASCADE,
    character_id TEXT NOT NULL REFERENCES characters(id) ON DELETE CASCADE,
    related_character_id TEXT NOT NULL REFERENCES characters(id) ON DELETE CASCADE,
    relationship_type TEXT NOT NULL,
    description TEXT,
    CHECK (character_id <> related_character_id)
);

CREATE TABLE illustrations (
    id TEXT PRIMARY KEY,
    story_id TEXT NOT NULL REFERENCES stories(id) ON DELETE CASCADE,
    storage_key TEXT NOT NULL,
    content_type TEXT NOT NULL,
    prompt TEXT,
    created_at TIMESTAMP NOT NULL
);

-- ============================================================
-- CHAPTERS AND SCENES
-- ============================================================

CREATE TABLE chapters (
    id TEXT PRIMARY KEY,
    story_id TEXT NOT NULL REFERENCES stories(id) ON DELETE CASCADE,
    number INTEGER NOT NULL,
    title TEXT NOT NULL,
    position INTEGER NOT NULL,
    UNIQUE (story_id, number),
    UNIQUE (story_id, position)
);

CREATE TABLE scenes (
    id TEXT PRIMARY KEY,
    chapter_id TEXT NOT NULL REFERENCES chapters(id) ON DELETE CASCADE,
    title TEXT NOT NULL,
    content TEXT,
    position INTEGER NOT NULL,
    location_id TEXT REFERENCES locations(id) ON DELETE SET NULL,
    illustration_id TEXT REFERENCES illustrations(id) ON DELETE RESTRICT,
    UNIQUE (chapter_id, position)
);

-- ============================================================
-- DIALOGUE
-- ============================================================

CREATE TABLE dialogue_lines (
    id TEXT PRIMARY KEY,
    scene_id TEXT NOT NULL REFERENCES scenes(id) ON DELETE CASCADE,
    speaker_character_id TEXT NOT NULL REFERENCES characters(id) ON DELETE CASCADE,
    text TEXT NOT NULL,
    position INTEGER NOT NULL,
    UNIQUE (scene_id, position)
);

-- ============================================================
-- SCENE RELATIONS
-- ============================================================

CREATE TABLE scene_characters (
    scene_id TEXT NOT NULL REFERENCES scenes(id) ON DELETE CASCADE,
    character_id TEXT NOT NULL REFERENCES characters(id) ON DELETE CASCADE,
    PRIMARY KEY (scene_id, character_id)
);

CREATE TABLE scene_artifacts (
    scene_id TEXT NOT NULL REFERENCES scenes(id) ON DELETE CASCADE,
    artifact_id TEXT NOT NULL REFERENCES artifacts(id) ON DELETE CASCADE,
    PRIMARY KEY (scene_id, artifact_id)
);

CREATE TABLE scene_relationships (
    scene_id TEXT NOT NULL REFERENCES scenes(id) ON DELETE CASCADE,
    relationship_id TEXT NOT NULL REFERENCES relationships(id) ON DELETE CASCADE,
    PRIMARY KEY (scene_id, relationship_id)
);

-- ============================================================
-- STORY PROJECTS
-- ============================================================

CREATE TABLE story_projects (
    id TEXT PRIMARY KEY,
    title TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'Draft',
    story_id TEXT UNIQUE REFERENCES stories(id) ON DELETE SET NULL,
    user_id TEXT
);

-- ============================================================
-- CREATIVE LIBRARIES
-- ============================================================

CREATE TABLE creative_libraries (
    id TEXT PRIMARY KEY,
    user_id TEXT
);

CREATE TABLE library_characters (
    id TEXT PRIMARY KEY,
    library_id TEXT NOT NULL REFERENCES creative_libraries(id) ON DELETE CASCADE,
    name TEXT NOT NULL,
    description TEXT,
    role TEXT
);

CREATE TABLE library_worlds (
    id TEXT PRIMARY KEY,
    library_id TEXT NOT NULL REFERENCES creative_libraries(id) ON DELETE CASCADE,
    name TEXT NOT NULL,
    description TEXT
);

CREATE TABLE library_locations (
    id TEXT PRIMARY KEY,
    library_id TEXT NOT NULL REFERENCES creative_libraries(id) ON DELETE CASCADE,
    name TEXT NOT NULL,
    description TEXT
);

CREATE TABLE library_artifacts (
    id TEXT PRIMARY KEY,
    library_id TEXT NOT NULL REFERENCES creative_libraries(id) ON DELETE CASCADE,
    name TEXT NOT NULL,
    description TEXT
);

-- ============================================================
-- INDEXES
-- ============================================================

CREATE INDEX idx_characters_story_id ON characters(story_id);
CREATE INDEX idx_locations_story_id ON locations(story_id);
CREATE INDEX idx_artifacts_story_id ON artifacts(story_id);
CREATE INDEX idx_relationships_story_id ON relationships(story_id);
CREATE INDEX idx_relationships_character_id ON relationships(character_id);
CREATE INDEX idx_relationships_related_character_id ON relationships(related_character_id);
CREATE INDEX idx_illustrations_story_id ON illustrations(story_id);
CREATE INDEX idx_chapters_story_id ON chapters(story_id);
CREATE INDEX idx_scenes_chapter_id ON scenes(chapter_id);
CREATE INDEX idx_scenes_location_id ON scenes(location_id);
CREATE INDEX idx_scenes_illustration_id ON scenes(illustration_id);
CREATE INDEX idx_dialogue_lines_scene_id ON dialogue_lines(scene_id);
CREATE INDEX idx_dialogue_lines_speaker_character_id ON dialogue_lines(speaker_character_id);
CREATE INDEX idx_scene_characters_character_id ON scene_characters(character_id);
CREATE INDEX idx_scene_artifacts_artifact_id ON scene_artifacts(artifact_id);
CREATE INDEX idx_scene_relationships_relationship_id ON scene_relationships(relationship_id);
CREATE INDEX idx_story_projects_user_id ON story_projects(user_id);
CREATE INDEX idx_creative_libraries_user_id ON creative_libraries(user_id);
CREATE INDEX idx_library_characters_library_id ON library_characters(library_id);
CREATE INDEX idx_library_worlds_library_id ON library_worlds(library_id);
CREATE INDEX idx_library_locations_library_id ON library_locations(library_id);
CREATE INDEX idx_library_artifacts_library_id ON library_artifacts(library_id);
"""

with connection:
    with connection.cursor() as cursor:
        cursor.execute(schema_query)

print("Database schema created successfully.")


Database schema created successfully.


In [14]:
with connection.cursor() as cursor:
    cursor.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_type = 'BASE TABLE'
        ORDER BY table_name;
    """)

    tables = cursor.fetchall()

for table in tables:
    print(table[0])

print(f"\nTotal tables: {len(tables)}")

ai_usage_records
artifacts
chapters
characters
creative_libraries
dialogue_lines
illustrations
library_artifacts
library_characters
library_locations
library_worlds
locations
relationships
scene_artifacts
scene_characters
scene_relationships
scenes
stories
story_projects
usages
worlds

Total tables: 21


In [2]:
with connection.cursor() as cursor:
    cursor.execute("""
        SELECT
            tc.table_name,
            kcu.column_name,
            ccu.table_name AS foreign_table,
            ccu.column_name AS foreign_column,
            rc.delete_rule
        FROM information_schema.table_constraints AS tc
        JOIN information_schema.key_column_usage AS kcu
            ON tc.constraint_name = kcu.constraint_name
            AND tc.constraint_schema = kcu.constraint_schema
        JOIN information_schema.constraint_column_usage AS ccu
            ON ccu.constraint_name = tc.constraint_name
            AND ccu.constraint_schema = tc.constraint_schema
        JOIN information_schema.referential_constraints AS rc
            ON rc.constraint_name = tc.constraint_name
            AND rc.constraint_schema = tc.constraint_schema
        WHERE tc.constraint_type = 'FOREIGN KEY'
          AND tc.table_schema = 'public'
        ORDER BY tc.table_name, kcu.column_name;
    """)

    foreign_keys = cursor.fetchall()

for fk in foreign_keys:
    print(f"{fk[0]}.{fk[1]} -> {fk[2]}.{fk[3]} | ON DELETE {fk[4]}")

print(f"\nTotal foreign keys: {len(foreign_keys)}")

artifacts.story_id -> stories.id | ON DELETE CASCADE
chapters.story_id -> stories.id | ON DELETE CASCADE
characters.story_id -> stories.id | ON DELETE CASCADE
dialogue_lines.scene_id -> scenes.id | ON DELETE CASCADE
dialogue_lines.speaker_character_id -> characters.id | ON DELETE CASCADE
illustrations.story_id -> stories.id | ON DELETE CASCADE
library_artifacts.library_id -> creative_libraries.id | ON DELETE CASCADE
library_characters.library_id -> creative_libraries.id | ON DELETE CASCADE
library_locations.library_id -> creative_libraries.id | ON DELETE CASCADE
library_worlds.library_id -> creative_libraries.id | ON DELETE CASCADE
locations.story_id -> stories.id | ON DELETE CASCADE
relationships.character_id -> characters.id | ON DELETE CASCADE
relationships.related_character_id -> characters.id | ON DELETE CASCADE
relationships.story_id -> stories.id | ON DELETE CASCADE
scene_artifacts.artifact_id -> artifacts.id | ON DELETE CASCADE
scene_artifacts.scene_id -> scenes.id | ON DELETE 

In [3]:
create_concept_tables_query = """
-- ============================================================
-- CONCEPTS
-- ============================================================

CREATE TABLE concepts (
    id TEXT PRIMARY KEY,
    project_id TEXT NOT NULL UNIQUE
        REFERENCES story_projects(id) ON DELETE CASCADE,

    story_idea TEXT,
    story_idea_source TEXT,
    story_idea_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    genre TEXT,
    genre_source TEXT,
    genre_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    world TEXT,
    world_source TEXT,
    world_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    main_characters TEXT,
    main_characters_source TEXT,
    main_characters_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    atmosphere TEXT,
    atmosphere_source TEXT,
    atmosphere_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    main_conflict TEXT,
    main_conflict_source TEXT,
    main_conflict_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    story_direction TEXT,
    story_direction_source TEXT,
    story_direction_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    status TEXT NOT NULL DEFAULT 'In Progress',

    CHECK (status IN ('In Progress', 'Ready')),

    CHECK (story_idea_source IS NULL OR story_idea_source IN ('user', 'ai')),
    CHECK (genre_source IS NULL OR genre_source IN ('user', 'ai')),
    CHECK (world_source IS NULL OR world_source IN ('user', 'ai')),
    CHECK (main_characters_source IS NULL OR main_characters_source IN ('user', 'ai')),
    CHECK (atmosphere_source IS NULL OR atmosphere_source IN ('user', 'ai')),
    CHECK (main_conflict_source IS NULL OR main_conflict_source IN ('user', 'ai')),
    CHECK (story_direction_source IS NULL OR story_direction_source IN ('user', 'ai'))
);

-- ============================================================
-- CONCEPT MESSAGES
-- ============================================================

CREATE TABLE concept_messages (
    id TEXT PRIMARY KEY,
    concept_id TEXT NOT NULL
        REFERENCES concepts(id) ON DELETE CASCADE,
    role TEXT NOT NULL,
    content TEXT NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,

    CHECK (role IN ('user', 'assistant'))
);

-- ============================================================
-- INDEXES
-- ============================================================

CREATE INDEX idx_concept_messages_concept_id
ON concept_messages(concept_id);
"""

with connection:
    with connection.cursor() as cursor:
        cursor.execute(create_concept_tables_query)

print("Concept tables created successfully.")


Concept tables created successfully.


In [5]:
create_blueprint_table_query = """
CREATE TABLE story_blueprints (
    id TEXT PRIMARY KEY,

    project_id TEXT NOT NULL UNIQUE
        REFERENCES story_projects(id) ON DELETE CASCADE,

    title_value TEXT NOT NULL,
    title_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    synopsis_source_value TEXT,
    synopsis_value TEXT,
    synopsis_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    genre_source_value TEXT,
    genre_value TEXT,
    genre_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    world_source_value TEXT,
    world_value TEXT,
    world_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    main_characters_source_value JSONB,
    main_characters_value JSONB,
    main_characters_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    atmosphere_source_value TEXT,
    atmosphere_value TEXT,
    atmosphere_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    main_conflict_source_value TEXT,
    main_conflict_value TEXT,
    main_conflict_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    plot_outline_source_value JSONB,
    plot_outline_value JSONB,
    plot_outline_accepted BOOLEAN NOT NULL DEFAULT FALSE,

    story_direction_source_value TEXT,
    story_direction_value TEXT,
    story_direction_accepted BOOLEAN NOT NULL DEFAULT FALSE
);
"""

with connection:
    with connection.cursor() as cursor:
        cursor.execute(create_blueprint_table_query)

print("Story Blueprint table created successfully.")

Story Blueprint table created successfully.


In [6]:
with connection.cursor() as cursor:
    cursor.execute("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = 'story_blueprints'
        ORDER BY ordinal_position;
    """)

    columns = cursor.fetchall()

for column in columns:
    print(column)

print("\nTotal columns:", len(columns))

('id', 'text')
('project_id', 'text')
('title_value', 'text')
('title_accepted', 'boolean')
('synopsis_source_value', 'text')
('synopsis_value', 'text')
('synopsis_accepted', 'boolean')
('genre_source_value', 'text')
('genre_value', 'text')
('genre_accepted', 'boolean')
('world_source_value', 'text')
('world_value', 'text')
('world_accepted', 'boolean')
('main_characters_source_value', 'jsonb')
('main_characters_value', 'jsonb')
('main_characters_accepted', 'boolean')
('atmosphere_source_value', 'text')
('atmosphere_value', 'text')
('atmosphere_accepted', 'boolean')
('main_conflict_source_value', 'text')
('main_conflict_value', 'text')
('main_conflict_accepted', 'boolean')
('plot_outline_source_value', 'jsonb')
('plot_outline_value', 'jsonb')
('plot_outline_accepted', 'boolean')
('story_direction_source_value', 'text')
('story_direction_value', 'text')
('story_direction_accepted', 'boolean')

Total columns: 28
